
# X-VECTOR FEATURE EXTRACTION PIPELINE
# Uses preprocessed patient-only WAV files and saves x-vectors


In [ ]:
# install packages
!pip install -U speechbrain librosa soundfile tqdm pandas


# mount Google Drive
from google.colab import drive
from pathlib import Path

# connect the colab runtime to google drive
drive.mount("/content/drive", force_remount=True)


# define locations
PROJECT = Path("/content/drive/MyDrive/asr project")
PREPROCESSED_ROOT = PROJECT / "preprocessed_patient_audio/Patients/"
FEATURE_ROOT = PROJECT / "features" / "xvector_patient_only"
assert PROJECT.exists(), f"Project folder not found: {PROJECT}"
assert PREPROCESSED_ROOT.exists(), f"Preprocessed folder not found: {PREPROCESSED_ROOT}"
FEATURE_ROOT.mkdir(parents=True, exist_ok=True)


print("Project:", PROJECT)
print("Preprocessed audio root:", PREPROCESSED_ROOT)
print("Feature output root:", FEATURE_ROOT)


# imports
import os
import numpy as np
import pandas as pd
import torch
import librosa
from tqdm import tqdm
from speechbrain.inference.classifiers import EncoderClassifier


# Load SpeechBrain x-vector model

# use gpu when available
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# load the pretrained speaker embedding model
classifier = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-xvect-voxceleb",
    savedir=str(PROJECT / "pretrained_models" / "spkrec-xvect-voxceleb"),
    run_opts={"device": device}
)


# audio loading function
def load_wav_mono_16k_librosa(wav_path, target_sr=16000):
    """
    Load WAV as mono, 16 kHz, float32, peak-normalized.
    The preprocessed files should already be mono/16k,
    but this keeps the feature extraction safe.
    """

    # load resample and convert the audio to mono
    x, sr = librosa.load(
        wav_path,
        sr=target_sr,
        mono=True
    )
    # use float32
    x = x.astype(np.float32)
    # find the highest absolute amplitude
    max_abs = np.max(np.abs(x)) if len(x) > 0 else 0.0
    # normalize non silent audio
    if max_abs > 0:
        x = x / max_abs
    # convert the waveform to a pytorch tensor
    return torch.tensor(x, dtype=torch.float32)


# x-vector extraction function
def extract_xvector(wav_path, classifier, device="cpu"):
    """
    Extract one x-vector from one WAV file.

    Returns:
        x-vector as numpy array
    """
    # load the processed audio waveform
    waveform = load_wav_mono_16k_librosa(wav_path)
    # add a batch dimension and move to the selected device
    waveform = waveform.unsqueeze(0).to(device)
    # disable gradient calculation during feature extraction
    with torch.no_grad():
        emb = classifier.encode_batch(waveform)
    # remove extra dimensions and convert to numpy
    emb = emb.squeeze().detach().cpu().numpy().astype(np.float32)
    return emb


# Find all patient-only WAV files
# search recursively for patient only audio files
patient_wavs = sorted(PREPROCESSED_ROOT.rglob("*_patient.wav"))

print("Found patient-only WAVs:", len(patient_wavs))
print("\nExample files:")
for p in patient_wavs[:10]:
    print(p)
# stop when no matching files are found
assert len(patient_wavs) > 0, "No *_patient.wav files found. Check preprocessing output."


#  еxtract and save x-vectors
# store one metadata row for each audio file
feature_rows = []

# process every patient recording
for wav_path in tqdm(patient_wavs):
    wav_path = Path(wav_path)

    # Expected structure:
    # preprocessed_patient_audio / DatasetName / wav_patient_only / file_patient.wav

    # identify the dataset and recording
    dataset_name = wav_path.parents[1].name
    file_id = wav_path.stem.replace("_patient", "")

    # create one output folder per dataset
    out_dir = FEATURE_ROOT / dataset_name
    out_dir.mkdir(parents=True, exist_ok=True)

    # define the saved embedding path
    out_path = out_dir / f"{file_id}_xvector.npy"

    try:
        # reuse an existing feature file
        if out_path.exists():
            xvec = np.load(out_path)
            status = "already_exists"
            error = ""

        else:
            # extract one x-vector
            xvec = extract_xvector(
                wav_path=wav_path,
                classifier=classifier,
                device=device
            )

            # save the vector as a numpy file
            np.save(out_path, xvec)

            status = "ok"
            error = ""

        # record successful extraction metadata
        feature_rows.append({
            "dataset": dataset_name,
            "file_id": file_id,
            "wav_path": str(wav_path),
            "xvector_path": str(out_path),
            "xvector_shape": str(tuple(xvec.shape)),
            "status": status,
            "error": error
        })

    # record errors without stopping the remaining files
    except Exception as e:
        feature_rows.append({
            "dataset": dataset_name,
            "file_id": file_id,
            "wav_path": str(wav_path),
            "xvector_path": str(out_path),
            "xvector_shape": "",
            "status": "error",
            "error": str(e)
        })



# convert extraction records to a dataframe
xvector_metadata = pd.DataFrame(feature_rows)

# save paths statuses and vector shapes
metadata_path = FEATURE_ROOT / "xvector_feature_metadata.csv"
xvector_metadata.to_csv(metadata_path, index=False)

# show extraction summary
print("\nSaved x-vector metadata to:", metadata_path)
print("Total files:", len(xvector_metadata))

# show successful existing and failed files
print("\nStatus counts:")
print(xvector_metadata["status"].value_counts())

# show status counts for each dataset
print("\nBy dataset/status:")
print(xvector_metadata.groupby(["dataset", "status"]).size())


# check one saved x-vector

# keep successfully extracted or previously saved vectors
valid_df = xvector_metadata[
    xvector_metadata["status"].isin(["ok", "already_exists"])
].copy()

# stop when no valid vectors exist
assert len(valid_df) > 0, "No valid x-vectors were extracted."

# load one vector for inspection
example_path = valid_df.iloc[0]["xvector_path"]
example_xvec = np.load(example_path)

print("\nExample x-vector path:", example_path)
print("Example x-vector shape:", example_xvec.shape)
print("First 10 values:")
print(example_xvec[:10])


# save features

# store one csv row per recording
rows = []

# load each valid x-vector
for _, row in valid_df.iterrows():
    xvec = np.load(row["xvector_path"])

    # add recording metadata
    feature_row = {
        "dataset": row["dataset"],
        "file_id": row["file_id"],
        "wav_path": row["wav_path"],
        "xvector_path": row["xvector_path"],
    }

    # save each vector value as a separate column
    for i, value in enumerate(xvec):
        feature_row[f"xvec_{i}"] = float(value)

    rows.append(feature_row)

# create the combined feature table
xvector_df = pd.DataFrame(rows)

# save all vectors in one csv
combined_csv_path = FEATURE_ROOT / "xvector_features_combined.csv"
xvector_df.to_csv(combined_csv_path, index=False)

print("\nSaved combined x-vector CSV to:", combined_csv_path)
print("Combined CSV shape:", xvector_df.shape)


# save combined matrix NPZ

X = []
dataset = []
file_id = []
wav_paths = []
xvector_paths = []

# load valid vectors in metadata order
for _, row in valid_df.iterrows():
    xvec = np.load(row["xvector_path"])

    X.append(xvec)
    dataset.append(row["dataset"])
    file_id.append(row["file_id"])
    wav_paths.append(row["wav_path"])
    xvector_paths.append(row["xvector_path"])

# stack vectors into one feature matrix
X = np.vstack(X).astype(np.float32)

# output path
npz_path = FEATURE_ROOT / "xvector_features_matrix.npz"

# save the matrix and aligned metadata
np.savez(
    npz_path,
    X=X,
    dataset=np.array(dataset),
    file_id=np.array(file_id),
    wav_path=np.array(wav_paths),
    xvector_path=np.array(xvector_paths)
)
print("\nSaved combined x-vector matrix to:", npz_path)
print("X shape:", X.shape)
print("\nDone.")
print("Feature folder:", FEATURE_ROOT)
print("Metadata CSV:", metadata_path)
print("Combined CSV:", combined_csv_path)
print("Combined NPZ:", npz_path)
```
